# NqText - Phase 3C: Chain-of-Thought Prompt

**Optimization:** Phase 3C - Chain-of-Thought prompting

**Dataset:** NqText (Wikipedia Factual Q&A)

**Model:** NVIDIA Nemotron-3 Ultra 550B (via Together AI)

**Metric:** Span F1

**Documents:** 3 example PDFs

**Q&A Count:** 78 pairs

**What changed:**
- ✅ Chain-of-Thought prompt (step-by-step reasoning)
- ✅ Keep all Phase 2 parameters (TOP_K=10, CHUNK_SIZE=3000)

**Baseline (Phase 2):**
- Empty rate: 7.7% (6/78 questions)

**Target:**
- Empty rate: <7.7% (any improvement)

**Expected runtime:** 39-58 minutes
**Expected cost:** 2x tokens

## Setup and Imports

In [1]:
import sys
import os

# Navigate to project root
project_root = os.path.abspath('../../../../..')
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


In [2]:
import pandas as pd
import re
import chromadb
import PyPDF2
import time
import importlib.util
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb.utils.embedding_functions as embedding_functions
from uda.utils import preprocess
from uda.utils.prompts import get_prompt
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

✓ All imports successful


/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configuration

In [3]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

Model: nvidia/nemotron-3-ultra-550b-a55b
API Key: tgp_v1_9OcdTuqoXTB0_...


In [4]:
# Experiment Parameters
DATASET_NAME = "nq"
CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300
TOP_K = 10
TEMPERATURE = 0.1
MAX_TOKENS = 512

# Prompt type
PROMPT_TYPE = "cot"

# Output settings
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = "./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/nqtext_cot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Prompt type: {PROMPT_TYPE}")
print(f"Output dir: {OUTPUT_DIR}")

Dataset: nq
Chunk size: 3000
Top-K: 10
Prompt type: cot
Output dir: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/nqtext_cot


## Initialize Models

In [5]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# Embedding model
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embedding model loaded: all-MiniLM-L6-v2")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

# Load prompt function
prompt_fn = get_prompt(PROMPT_TYPE)
print(f"✓ Prompt function loaded: {PROMPT_TYPE}")

✓ Together AI client initialized


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model loaded: all-MiniLM-L6-v2
✓ Text splitter initialized
✓ Prompt function loaded: cot


## Helper Functions

In [6]:
def extract_pdf_text(pdf_path):
    """Extract text from PDF using PyPDF2"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def sanitize_collection_name(doc_name, dataset_name):
    """
    Sanitize document name for ChromaDB collection.
    
    ChromaDB requires: 3-512 chars from [a-zA-Z0-9._-], 
    starting and ending with alphanumeric.
    """
    # Replace invalid chars with underscore
    safe_name = re.sub(r'[^a-zA-Z0-9._-]', '_', doc_name)
    # Remove consecutive underscores
    safe_name = re.sub(r'_+', '_', safe_name)
    # Remove leading/trailing underscores
    safe_name = safe_name.strip('_')
    # Prepend dataset name
    collection_name = f"{dataset_name}_{safe_name}"
    return collection_name

def build_index(text_chunks, collection_name="temp_collection"):
    """Build vector index"""
    chroma_client = chromadb.Client()

    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass

    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )

    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)

    return collection

def answer_question(collection, question):
    """
    Retrieve context and generate answer.

    CHANGED: Uses PROMPT_TYPE prompt
    """
    # Retrieve
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])

    # Build prompt using prompts module
    prompt_text = prompt_fn(context=context, question=question)

    # Convert to message format
    messages = [
        {"role": "user", "content": prompt_text}
    ]

    # Generate
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    return response.choices[0].message.content

print("✓ Helper functions defined")


✓ Helper functions defined


## Load Q&A Data

In [7]:
# Load Q&A
csv_file = "./dataset/qa/nq_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict_all = preprocess.qa_df_to_dict(DATASET_NAME, df)

# Filter to documents with available PDFs - PHASE 2 DOCUMENT LIST
AVAILABLE_DOCS = [
    "2018 Tour de France",
    "Hannah John-Kamen",
    "Oklahoma",  # ADDED: Phase 2 included this (7 Q&A)
    "Supreme Court of the United States"
]

qas_dict = {doc: qas for doc, qas in qas_dict_all.items() if doc in AVAILABLE_DOCS}

print(f"Total documents in CSV: {len(qas_dict_all)}")
print(f"Available PDFs: {len(AVAILABLE_DOCS)}")
print(f"\nFiltered to documents with PDFs:\n")

total_qa = 0
for doc in AVAILABLE_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"  {doc}: {count} Q&A pairs")

print(f"\nTotal Q&A to process: {total_qa}")


Total documents in CSV: 645
Available PDFs: 4

Filtered to documents with PDFs:

  2018 Tour de France: 13 Q&A pairs
  Hannah John-Kamen: 1 Q&A pairs
  Oklahoma: 7 Q&A pairs
  Supreme Court of the United States: 57 Q&A pairs

Total Q&A to process: 78


## Main Processing Loop

**Expected runtime:** 39-58 minutes

In [8]:
all_results = []

for doc_name, doc_qas in qas_dict.items():
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")

    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, doc_name)
    if not pdf_path:
        print(f"❌ PDF not found - skipping")
        continue

    print(f"PDF: {pdf_path}")

    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")

    # Build index
    print("Building vector index...")
    collection_name = sanitize_collection_name(doc_name, DATASET_NAME)
    collection = build_index(text_chunks, collection_name=collection_name)
    print("✓ Index built")

    # Process questions
    print(f"\nAnswering {len(doc_qas)} questions...")

    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")

        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")

            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": DATASET_NAME,
                "prompt_type": PROMPT_TYPE,
            })

            time.sleep(0.5)

        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue

    print(f"\n✓ Completed {doc_name}")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")



Processing: Supreme Court of the United States
PDF: dataset/src_doc_files_example/wiki_nq_docs/pdfs/Supreme Court of the United States.pdf
Extracting text...
Created 67 chunks
Building vector index...
✓ Index built

Answering 57 questions...

[1/57] who determines the size of the supreme court...
   Answer: Based on the provided context, **Congress determines the size of the Supreme Cou...

[2/57] where is the supreme court of the united states located...
   Answer: The Supreme Court of the United States is located in Washington, D.C., United St...

[3/57] who is the supreme court made up of...
   Answer: Based on the provided context, the Supreme Court of the United States is made up...

[4/57] what is the highest court in the united states...
   Answer: The Supreme Court of the United States is the highest federal court in the Unite...

[5/57] how long has the supreme court has 9 justices...
   Answer: Based on the provided context, the Supreme Court has had 9 justices since **1869.

## Diagnostic: Check Empty Responses

In [9]:
if all_results:
    results_df = pd.DataFrame(all_results)

    # Count empty responses
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    empty_count = results_df['is_empty'].sum()
    total_count = len(results_df)

    print(f"\n{'='*80}")
    print(f"DIAGNOSTIC: Empty Response Analysis")
    print(f"{'='*80}")
    print(f"Total Q&A processed: {total_count}")
    print(f"Empty responses: {empty_count} ({empty_count/total_count*100:.1f}%)")
    print(f"Answered: {total_count - empty_count} ({(total_count-empty_count)/total_count*100:.1f}%)")

    # Comparison with Phase 2
    phase2_empty = 6
    phase2_total = 78
    phase2_empty_pct = 7.7

    improvement = phase2_empty - empty_count
    improvement_pct = phase2_empty_pct - (empty_count/total_count*100)

    print(f"\n{'='*80}")
    print(f"COMPARISON WITH PHASE 2 BASELINE")
    print(f"{'='*80}")
    print(f"Phase 2 (Baseline): {phase2_empty}/{phase2_total} empty ({phase2_empty_pct:.1f}%)")
    print(f"Phase 3C (Chain-of-Thought): {empty_count}/{total_count} empty ({empty_count/total_count*100:.1f}%)")
    print(f"\nImprovement: {improvement:+d} questions ({improvement_pct:+.1f} percentage points)")

    if improvement > 0:
        print(f"✅ SUCCESS: Chain-of-Thought reduced empty responses!")
    elif improvement == 0:
        print(f"⚠️  NEUTRAL: No change")
    else:
        print(f"❌ REGRESSION: Empty responses increased")

    if empty_count > 0:
        print(f"\nEmpty responses by document:")
        for doc in results_df['doc'].unique():
            doc_df = results_df[results_df['doc'] == doc]
            doc_empty = doc_df['is_empty'].sum()
            doc_total = len(doc_df)
            print(f"  {doc}: {doc_empty}/{doc_total} empty ({doc_empty/doc_total*100:.1f}%)")
else:
    print("❌ No results to analyze")


DIAGNOSTIC: Empty Response Analysis
Total Q&A processed: 78
Empty responses: 11 (14.1%)
Answered: 67 (85.9%)

COMPARISON WITH PHASE 2 BASELINE
Phase 2 (Baseline): 6/78 empty (7.7%)
Phase 3C (Chain-of-Thought): 11/78 empty (14.1%)

Improvement: -5 questions (-6.4 percentage points)
❌ REGRESSION: Empty responses increased

Empty responses by document:
  Supreme Court of the United States: 7/57 empty (12.3%)
  2018 Tour de France: 3/13 empty (23.1%)
  Hannah John-Kamen: 0/1 empty (0.0%)
  Oklahoma: 1/7 empty (14.3%)


## Evaluate Results

In [10]:
if all_results:
    print(f"\nEvaluating {DATASET_NAME} results...")
    eval_main(DATASET_NAME, all_results)
else:
    print("❌ No results to evaluate")


Evaluating nq results...
{'Answer F1': 0.25456587111476886, 'Missing predictions': 0}


## Save Results

In [11]:
if all_results:
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_DIR, f"nqtext_cot_{TIMESTAMP}.csv")
    results_df.to_csv(output_file, index=False)

    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")

    print("\nResults by document:")
    for doc in results_df['doc'].unique():
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")


✓ Results saved to: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/nqtext_cot/nqtext_cot_20260630_110945.csv
Total Q&A: 78

Results by document:
  Supreme Court of the United States: 57 questions
  2018 Tour de France: 13 questions
  Hannah John-Kamen: 1 questions
  Oklahoma: 7 questions


## Final Summary

In [12]:
if all_results:
    results_df = pd.DataFrame(all_results)

    empty_count = results_df['response'].fillna('').str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count

    phase2_empty = 6
    improvement = phase2_empty - empty_count

    print(f"\n{'='*80}")
    print(f"FINAL SUMMARY - CHAIN-OF-THOUGHT (NqText)")
    print(f"{'='*80}")
    print(f"Dataset: NqText ({len(results_df)} Q&A)")
    print(f"Prompt type: {PROMPT_TYPE}")
    print(f"\nResults:")
    print(f"  Answered: {answered_count}/{len(results_df)} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"  Empty: {empty_count}/{len(results_df)} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"\nVs Phase 2 Baseline:")
    print(f"  Change: {improvement:+d} questions")
    print(f"  Cost: 2x tokens")

    if improvement >= 2:
        print(f"\n✅ EXCELLENT: Chain-of-Thought significantly improved!")
    elif improvement >= 1:
        print(f"\n✅ GOOD: Chain-of-Thought helped")
    elif improvement == 0:
        print(f"\n⚠️  NEUTRAL: No change")
    else:
        print(f"\n❌ REGRESSION: Made things worse")
else:
    print("\n❌ No results to summarize")


FINAL SUMMARY - CHAIN-OF-THOUGHT (NqText)
Dataset: NqText (78 Q&A)
Prompt type: cot

Results:
  Answered: 67/78 (85.9%)
  Empty: 11/78 (14.1%)

Vs Phase 2 Baseline:
  Change: -5 questions
  Cost: 2x tokens

❌ REGRESSION: Made things worse


---

## Done!

**Results saved to:** `./results/nqtext_cot/`

Compare with other prompt types to find the best approach for NqText.